# Notebook 47 — Probe-Gated Memory (Paper-5 Standalone Proof)

**Question**: does FG+RG ensemble-gated memory bank improve accuracy over zero-shot, on multi-hop QA OOD for both probes?

Probe-only test — no LLM judge. Goal: prove probes work as memory gates, period.

## Setup

- Base Qwen3.6-27B (no LoRA, inference middleware)
- HotpotQA distractor: n=60 train + n=60 test (multi-hop, OOD for both probes)
- Memory unit: `{question, reasoning_chain, answer, embedding}`
- Embedder: sentence-transformers/all-MiniLM-L6-v2
- Retrieval: top-3 nearest by question cosine similarity

## 4 conditions

| Condition | Gate | Probes the |
|---|---|---|
| **none** | (no memory) | baseline |
| **ensemble-gated** | `(fg+rg)/2 < 0.5` → admit | core claim |
| **all-admit** | every trajectory in | quality vs quantity control |
| **random-50%** | random | sanity (must beat random) |

## Verdict criteria

| Verdict | Criterion |
|---|---|
| 🟢 STRONG | ensemble > random > none, paired CI excludes 0 |
| 🟢 STANDARD | ensemble > none AND ensemble ≥ random |
| 🟡 MIXED | ensemble > none but ≈ random |
| 🔴 KILL | ensemble ≤ none |

## Compute

~5h on RTX 6000 (60 train + 4×60 test gens). $0 in API calls (no judge).

**Drive**: `/content/drive/MyDrive/openinterp_runs/47_probe_gated_memory/`

## Free post-hoc

Same train pool gives FG-only / RG-only / weighted_avg / max / bayesian_or gate variants for free (only memory-bank rebuild, no re-gen). Supplementary if main result passes.


## Phase 1 — Setup + Drive


In [ ]:
from pathlib import Path
import os, json, time
import torch, numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / '47_probe_gated_memory'
OUT.mkdir(parents=True, exist_ok=True)
print(f'OUT: {OUT}')
print(f'Existing files: {sorted(p.name for p in OUT.iterdir())}')


In [ ]:
!pip install -q -U torchao
!pip install -q -U transformers accelerate datasets
!pip install -q -U huggingface_hub sentence-transformers
!pip install -q scikit-learn joblib matplotlib
print('✓ deps')


## Phase 2 — Qwen3.6-27B + FG + RG probes + sentence embedder


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, hf_hub_download, HfApi, create_repo
from sentence_transformers import SentenceTransformer
import getpass, joblib

CFG = {
    'model_id':              'Qwen/Qwen3.6-27B',
    'capture_layer_fg':      31,
    'capture_layer_rg':      55,
    'n_train':               60,
    'n_test':                60,
    'k_retrieve':            3,
    'gate_threshold':        0.5,
    'temperature':           0.7,
    'max_new_tokens':        2048,
    'random_seed':           47,
    'fg_probe_repo':         'caiovicentino1/FabricationGuard-linearprobe-qwen36-27b',
    'rg_probe_repo':         'caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b',
    'embedder_model':        'sentence-transformers/all-MiniLM-L6-v2',
    'output_repo':           'caiovicentino1/openinterp-47-probe-gated-memory',
    'bootstrap_n':           1000,
    'conditions':            ['none', 'ensemble-gated', 'all-admit', 'random-50'],
}
THINK_CLOSE_ID = 248069
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda'
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto',
)
model.eval()
print(f'✓ Base loaded — {torch.cuda.get_device_name(0)}')


In [ ]:
fg_path = hf_hub_download(repo_id=CFG['fg_probe_repo'], filename='probe.joblib', repo_type='dataset')
rg_path = hf_hub_download(repo_id=CFG['rg_probe_repo'], filename='probe.joblib', repo_type='dataset')
fg_artifact = joblib.load(fg_path)
rg_artifact = joblib.load(rg_path)
fg_clf = fg_artifact['probe']; fg_scaler = fg_artifact['scaler']
rg_clf = rg_artifact['probe']; rg_scaler = rg_artifact['scaler']
if not hasattr(fg_clf, 'multi_class'): fg_clf.multi_class = 'auto'
if not hasattr(rg_clf, 'multi_class'): rg_clf.multi_class = 'auto'

def fg_score(act):
    x = act.float().cpu().numpy().reshape(1, -1)
    return float(fg_clf.predict_proba(fg_scaler.transform(x))[0, 1])
def rg_score(act):
    x = act.float().cpu().numpy().reshape(1, -1)
    return float(rg_clf.predict_proba(rg_scaler.transform(x))[0, 1])
print('✓ FG + RG probes loaded')

embedder = SentenceTransformer(CFG['embedder_model'], device='cuda')
print(f'✓ Embedder loaded: dim={embedder.get_sentence_embedding_dimension()}')


## Phase 3 — Hooks + HotpotQA load


In [ ]:
captured = {}
_pos = {'pos': None}

def make_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        pos = _pos['pos']
        if pos is None or pos >= h.shape[1]: return
        captured[f'L{layer_idx}'] = h[0, pos, :].detach().cpu().to(torch.float16).clone()
    return hook

hook_handles = []
for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
    h = model.model.layers[L].register_forward_hook(make_hook(L))
    hook_handles.append(h)
print(f'✓ Hooks at L{CFG["capture_layer_fg"]}, L{CFG["capture_layer_rg"]}')


In [ ]:
from datasets import load_dataset
rng = np.random.default_rng(CFG['random_seed'])

hpqa = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation', trust_remote_code=True)
all_idx = rng.choice(len(hpqa), size=CFG['n_train'] + CFG['n_test'], replace=False)
train_idx = all_idx[:CFG['n_train']]
test_idx = all_idx[CFG['n_train']:]

def to_record(ex, i, split):
    return {
        'id': f'{split}_{i}',
        'split': split,
        'question': ex['question'],
        'gold': ex['answer'],
        'type': ex.get('type', ''),
        'level': ex.get('level', ''),
    }

train_pool = [to_record(hpqa[int(i)], int(i), 'train') for i in train_idx]
test_pool = [to_record(hpqa[int(i)], int(i), 'test') for i in test_idx]
print(f'Train: {len(train_pool)} | Test: {len(test_pool)}')
print(f'Sample question: {train_pool[0]["question"]}')
print(f'Sample gold: {train_pool[0]["gold"]}')


## Phase 4 — Generate train pool with FG + RG capture

Each train question: generate trajectory at temp=0.7, capture L31 + L55 at end-of-think.
Resume-safe via Drive jsonl append.


In [ ]:
from tqdm.auto import tqdm
import gc

def find_end_think(token_ids):
    ids = token_ids.tolist() if hasattr(token_ids, 'tolist') else list(token_ids)
    for i in range(len(ids) - 1, -1, -1):
        if ids[i] == THINK_CLOSE_ID: return i
    return None

def generate_with_memories(question, memories=None):
    """Generate at temp=0.7, optionally with retrieved memories in context."""
    if memories:
        mem_text = '\n\n'.join([
            f'Example {i+1}:\nQ: {m["question"]}\nReasoning: {m["reasoning"][:800]}\nA: {m["answer"]}'
            for i, m in enumerate(memories)
        ])
        user_msg = f'Here are similar questions with reasoning:\n\n{mem_text}\n\nNow answer this question:\nQ: {question}'
    else:
        user_msg = question
    messages = [{'role': 'user', 'content': user_msg}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(text, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    amask = enc.get('attention_mask', torch.ones_like(ids)).to(device)
    n_in = ids.shape[1]
    with torch.no_grad():
        gen = model.generate(
            ids, attention_mask=amask,
            max_new_tokens=CFG['max_new_tokens'],
            do_sample=True, temperature=CFG['temperature'], top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    full_ids = gen[0]
    output_ids = full_ids[n_in:]
    end_pos = find_end_think(full_ids)
    output_text = tok.decode(output_ids, skip_special_tokens=False)
    if '</think>' in output_text:
        cot = output_text.split('</think>', 1)[0].strip()
        answer = output_text.split('</think>', 1)[1].strip()
    else:
        cot, answer = output_text.strip(), ''
    return {
        'cot': cot, 'answer': answer,
        'has_think': end_pos is not None, 'end_pos': end_pos,
        'full_ids': full_ids,
    }

def score_trajectory(full_ids, end_pos):
    """Replay forward at end-of-think to score with FG + RG."""
    if end_pos is None:
        return None, None
    captured.clear()
    _pos['pos'] = end_pos
    with torch.no_grad():
        _ = model(full_ids.unsqueeze(0).to(device))
    act_fg = captured.get(f'L{CFG["capture_layer_fg"]}')
    act_rg = captured.get(f'L{CFG["capture_layer_rg"]}')
    fg = fg_score(act_fg) if act_fg is not None else None
    rg = rg_score(act_rg) if act_rg is not None else None
    return fg, rg

train_path = OUT / 'train_pool.jsonl'
done_train = set()
if train_path.exists():
    with open(train_path) as f:
        for line in f:
            try: done_train.add(json.loads(line)['id'])
            except: continue
    print(f'Resume train: {len(done_train)} done')

for p in tqdm(train_pool, desc='train gen+probe'):
    if p['id'] in done_train: continue
    try:
        torch.manual_seed(hash(p['id']) % (2**32))
        res = generate_with_memories(p['question'])
        fg, rg = score_trajectory(res['full_ids'], res['end_pos'])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        print(f'OOM on {p["id"]}'); continue
    record = {
        **p,
        'reasoning': res['cot'],
        'answer': res['answer'],
        'has_think': res['has_think'],
        'fg': fg, 'rg': rg,
        'ensemble': (fg + rg) / 2 if (fg is not None and rg is not None) else None,
    }
    record.pop('full_ids', None)
    with open(train_path, 'a') as f:
        f.write(json.dumps(record) + '\n')
    done_train.add(p['id'])

print('\n✓ Phase 4 complete')


## Phase 5 — Build 4 memory banks with embeddings

Same train trajectories, 4 different gates → 4 memory banks of different sizes.
Embeddings computed once on questions, shared across banks.


In [ ]:
with open(train_path) as f:
    train_records = [json.loads(line) for line in f]

# Filter to records with valid probe scores (need has_think + both probes)
valid_train = [r for r in train_records if r.get('has_think') and r.get('fg') is not None and r.get('rg') is not None]
print(f'Valid train trajectories: {len(valid_train)} / {len(train_records)}')

# Embed all train questions once
train_questions = [r['question'] for r in valid_train]
train_embeds = embedder.encode(train_questions, normalize_embeddings=True, show_progress_bar=True)
print(f'Train embeddings: {train_embeds.shape}')

# Compute gate decisions
rng_gate = np.random.default_rng(CFG['random_seed'] + 1)
for i, r in enumerate(valid_train):
    r['embed_idx'] = i
    r['gate_ensemble'] = r['ensemble'] < CFG['gate_threshold']
    r['gate_random'] = rng_gate.random() < 0.5
    r['gate_all'] = True

# Build banks
banks = {
    'none': [],
    'ensemble-gated': [r for r in valid_train if r['gate_ensemble']],
    'all-admit':      [r for r in valid_train if r['gate_all']],
    'random-50':      [r for r in valid_train if r['gate_random']],
}
for name, bank in banks.items():
    print(f'  {name}: {len(bank)} memories')

# Save bank index for resume
bank_meta = {name: [r['id'] for r in bank] for name, bank in banks.items()}
(OUT / 'memory_banks.json').write_text(json.dumps(bank_meta, indent=2))

# Sanity: ensemble score distribution
ens_scores = np.array([r['ensemble'] for r in valid_train])
fg_scores = np.array([r['fg'] for r in valid_train])
rg_scores = np.array([r['rg'] for r in valid_train])
print(f'\nEnsemble score quantiles: 25%={np.percentile(ens_scores, 25):.3f}  50%={np.percentile(ens_scores, 50):.3f}  75%={np.percentile(ens_scores, 75):.3f}')
print(f'FG-RG correlation in train pool: {np.corrcoef(fg_scores, rg_scores)[0,1]:+.3f}')


In [ ]:
def retrieve(question_embed, bank, k=3):
    if not bank: return []
    bank_embeds = train_embeds[[r['embed_idx'] for r in bank]]
    sims = bank_embeds @ question_embed
    top_k = np.argsort(-sims)[:k]
    return [bank[int(i)] for i in top_k]

# Smoke test retrieval on first test question
q_test = test_pool[0]['question']
q_embed = embedder.encode([q_test], normalize_embeddings=True)[0]
for name, bank in banks.items():
    if not bank: continue
    r = retrieve(q_embed, bank, k=CFG['k_retrieve'])
    print(f'{name}: top-1 retrieval for test_q={q_test[:80]}...')
    print(f'  → {r[0]["question"][:80]}...' if r else '  (empty)')


## Phase 6 — Test pool with retrieval per condition

For each test question × each condition: retrieve top-3 memories from that condition's bank,
feed to model with question, gen with thinking, store answer.

Resume-safe per (test_id, condition) pair. ~4h for 4 × 60 = 240 generations.


In [ ]:
test_path = OUT / 'test_pool.jsonl'
done_test = set()
if test_path.exists():
    with open(test_path) as f:
        for line in f:
            try:
                rec = json.loads(line)
                done_test.add((rec['id'], rec['condition']))
            except: continue
    print(f'Resume test: {len(done_test)} (id, condition) done')

# Pre-embed test questions
test_questions = [p['question'] for p in test_pool]
test_embeds = embedder.encode(test_questions, normalize_embeddings=True, show_progress_bar=True)

total = len(test_pool) * len(CFG['conditions'])
pbar = tqdm(total=total - len(done_test), desc='test gen')

for cond in CFG['conditions']:
    bank = banks[cond]
    for ti, p in enumerate(test_pool):
        key = (p['id'], cond)
        if key in done_test:
            continue
        try:
            torch.manual_seed(hash(p['id']) % (2**32))
            if cond == 'none' or not bank:
                memories = []
            else:
                memories = retrieve(test_embeds[ti], bank, k=CFG['k_retrieve'])
            res = generate_with_memories(p['question'], memories=memories or None)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            pbar.update(1); continue
        record = {
            **p,
            'condition': cond,
            'memories_used': [m['id'] for m in (memories or [])],
            'memories_count': len(memories or []),
            'reasoning': res['cot'],
            'answer': res['answer'],
            'has_think': res['has_think'],
        }
        with open(test_path, 'a') as f:
            f.write(json.dumps(record) + '\n')
        done_test.add(key)
        pbar.update(1)
        if pbar.n % 20 == 0:
            torch.cuda.empty_cache(); gc.collect()
pbar.close()
print('\n✓ Phase 6 complete')


## Phase 7 — F1 scoring + paired bootstrap CI

HotpotQA canonical metrics: F1 token overlap + exact match. No LLM judge needed.


In [ ]:
import re, string
from collections import Counter

def normalize_answer(s):
    """HotpotQA standard normalization."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0.0
    precision = num_same / len(pred_tokens) if pred_tokens else 0
    recall = num_same / len(gold_tokens) if gold_tokens else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def exact_match(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

def extract_answer(text):
    """Extract final answer from generated text. Take last non-empty line, strip common prefixes."""
    if not text: return ''
    text = text.strip()
    # Common patterns: 'Answer:', 'A:', etc.
    for prefix in ['Answer:', 'A:', 'answer:', 'Final answer:', 'The answer is']:
        if prefix in text:
            text = text.split(prefix, 1)[-1].strip()
            break
    # Take first 200 chars as the answer (HotpotQA answers are short)
    return text[:200].strip().split('\n')[0]

with open(test_path) as f:
    test_records = [json.loads(line) for line in f]

import pandas as pd
tdf = pd.DataFrame(test_records)
tdf['extracted'] = tdf['answer'].apply(extract_answer)
tdf['f1'] = tdf.apply(lambda r: f1_score(r['extracted'], r['gold']), axis=1)
tdf['em'] = tdf.apply(lambda r: float(exact_match(r['extracted'], r['gold'])), axis=1)

print(f'Total test records: {len(tdf)}')
print(f'has_think rate per condition:')
print(tdf.groupby('condition')['has_think'].mean().round(3))
print(f'\nF1 per condition:')
print(tdf.groupby('condition')['f1'].agg(['mean', 'std', 'count']).round(4))
print(f'\nExact match per condition:')
print(tdf.groupby('condition')['em'].agg(['mean', 'std']).round(4))


In [ ]:
# Paired bootstrap CI on F1 delta vs none
def paired_delta_ci(df_a, df_b, metric='f1', n=1000, seed=42):
    rng = np.random.default_rng(seed)
    merged = df_a.merge(df_b, on='id', suffixes=('_a', '_b'))
    da = merged[f'{metric}_a'].values
    db = merged[f'{metric}_b'].values
    deltas = []
    for _ in range(n):
        idx = rng.choice(len(da), size=len(da), replace=True)
        deltas.append(db[idx].mean() - da[idx].mean())
    arr = np.array(deltas)
    return float(arr.mean()), float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

none_df = tdf[tdf['condition'] == 'none'][['id', 'f1', 'em']]
results = {}
for cond in CFG['conditions']:
    cond_df = tdf[tdf['condition'] == cond][['id', 'f1', 'em']]
    cond_f1 = cond_df['f1'].mean()
    cond_em = cond_df['em'].mean()
    if cond == 'none':
        results[cond] = {'f1': cond_f1, 'em': cond_em, 'delta_f1': 0.0,
                         'delta_ci_lo': 0.0, 'delta_ci_hi': 0.0,
                         'memory_size': 0}
        continue
    delta, lo, hi = paired_delta_ci(none_df, cond_df, metric='f1', n=CFG['bootstrap_n'])
    results[cond] = {
        'f1': float(cond_f1),
        'em': float(cond_em),
        'delta_f1': delta,
        'delta_ci_lo': lo,
        'delta_ci_hi': hi,
        'memory_size': len(banks[cond]),
    }

print(f'\n{"condition":<20} {"F1":>8} {"ΔF1 vs none":>14} {"95% CI":>22} {"mem":>5}')
print('-' * 75)
for cond, r in results.items():
    ci = f'[{r["delta_ci_lo"]:+.3f}, {r["delta_ci_hi"]:+.3f}]' if cond != 'none' else ''
    delta_str = f'{r["delta_f1"]:+.4f}' if cond != 'none' else ''
    print(f'{cond:<20} {r["f1"]:>8.4f} {delta_str:>14} {ci:>22} {r["memory_size"]:>5}')


In [ ]:
# Aggregate verdict
ens = results['ensemble-gated']
rnd = results['random-50']
all_a = results['all-admit']
none_r = results['none']

ens_beats_none = (ens['delta_f1'] > 0) and (ens['delta_ci_lo'] > 0)
ens_beats_random = ens['f1'] > rnd['f1']
ens_at_least_random = ens['f1'] >= rnd['f1'] - 0.005
random_beats_none = (rnd['delta_f1'] > 0) and (rnd['delta_ci_lo'] > 0)

if ens_beats_none and ens_beats_random and not random_beats_none:
    verdict = '🟢 STRONG — probe-gated > random > none. Probe gate adds quality, not just quantity.'
    paper5_status = 'ship'
elif ens_beats_none and ens_at_least_random:
    verdict = '🟢 STANDARD — probe-gated > none AND probe-gated ≥ random. Core claim holds.'
    paper5_status = 'ship_with_scope'
elif ens_beats_none and not ens_at_least_random:
    verdict = '🟡 MIXED — probe-gated > none but ≤ random. Memory helps but probe filter is weak.'
    paper5_status = 'weak'
else:
    verdict = '🔴 KILL — probe-gated ≤ none. Probe gate not viable as memory filter on this task.'
    paper5_status = 'rejected'

print(f'\n=== VERDICT ===')
print(f'{verdict}')
print(f'paper-5 (Probe-Gated Memory) status: {paper5_status}')
print(f'\nKey deltas:')
print(f'  ensemble vs none:    F1={ens["delta_f1"]:+.4f}  CI=[{ens["delta_ci_lo"]:+.3f}, {ens["delta_ci_hi"]:+.3f}]')
print(f'  random vs none:      F1={rnd["delta_f1"]:+.4f}  CI=[{rnd["delta_ci_lo"]:+.3f}, {rnd["delta_ci_hi"]:+.3f}]')
print(f'  all-admit vs none:   F1={all_a["delta_f1"]:+.4f}  CI=[{all_a["delta_ci_lo"]:+.3f}, {all_a["delta_ci_hi"]:+.3f}]')
print(f'  ensemble vs random:  F1={ens["f1"] - rnd["f1"]:+.4f}')


## Phase 8 — Visualization + final verdict + push


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
conds = list(results.keys())
f1s = [results[c]['f1'] for c in conds]
deltas = [results[c]['delta_f1'] for c in conds]
lo_errs = [results[c]['delta_f1'] - results[c]['delta_ci_lo'] for c in conds]
hi_errs = [results[c]['delta_ci_hi'] - results[c]['delta_f1'] for c in conds]
mem_sizes = [results[c]['memory_size'] for c in conds]

color_map = {
    'none': '#9ca3af',
    'ensemble-gated': '#10b981',
    'all-admit': '#3b82f6',
    'random-50': '#f59e0b',
}
colors = [color_map.get(c, '#9ca3af') for c in conds]

axes[0].bar(conds, f1s, color=colors, alpha=0.85)
axes[0].set_ylabel('F1 (HotpotQA)')
axes[0].set_title('F1 per condition (probe-gated standalone proof)')
axes[0].grid(alpha=0.3)
for i, (c, f1, m) in enumerate(zip(conds, f1s, mem_sizes)):
    axes[0].text(i, f1 + 0.005, f'{f1:.3f}\n(mem={m})', ha='center', fontsize=9)

non_none_idx = [i for i, c in enumerate(conds) if c != 'none']
axes[1].bar([conds[i] for i in non_none_idx],
            [deltas[i] for i in non_none_idx],
            yerr=[[lo_errs[i] for i in non_none_idx], [hi_errs[i] for i in non_none_idx]],
            color=[colors[i] for i in non_none_idx], alpha=0.85, capsize=8)
axes[1].axhline(0, color='black', alpha=0.5)
axes[1].set_ylabel('ΔF1 vs none (paired bootstrap)')
axes[1].set_title('Memory effect: lift over zero-shot baseline')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'fig_probe_gated_memory.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
verdict_obj = {
    'experiment': 'nb47 probe-gated memory standalone proof',
    'hypothesis': 'FG+RG ensemble-gated memory bank improves accuracy over zero-shot',
    'task': 'hotpotqa-distractor (validation)',
    'n_train': len(valid_train),
    'n_test': len(test_pool),
    'k_retrieve': CFG['k_retrieve'],
    'gate_threshold': CFG['gate_threshold'],
    'memory_bank_sizes': {c: results[c]['memory_size'] for c in conds},
    'results_per_condition': results,
    'verdict': verdict,
    'paper5_status': paper5_status,
    'fg_rg_correlation_train': float(np.corrcoef(fg_scores, rg_scores)[0,1]),
}
(OUT / 'FINAL_VERDICT.json').write_text(json.dumps(verdict_obj, indent=2, default=str))
print(json.dumps(verdict_obj, indent=2, default=str))


In [ ]:
api = HfApi()
try: create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e: print(e)

readme_lines = [
    '---',
    'license: apache-2.0',
    'tags:',
    '- probe-gated-memory',
    '- inference-middleware',
    '- qwen36-27b',
    '- hotpotqa',
    '- paper-5',
    '---',
    '',
    '# nb47 — Probe-Gated Memory: Standalone Proof',
    '',
    f'**Verdict**: {verdict}',
    '',
    f'paper-5 (Probe-Gated Memory) status: **{paper5_status}**',
    '',
    '## Question',
    '',
    'Does FG+RG ensemble-gated memory bank improve downstream accuracy over zero-shot, on multi-hop QA OOD for both probes? **No LLM judge involved.**',
    '',
    '## Setup',
    '',
    f'- Base Qwen3.6-27B (no LoRA)',
    f'- HotpotQA distractor: {len(valid_train)} train pool + {len(test_pool)} test pool',
    f'- Memory unit: {{question, reasoning, answer, embedding}}',
    f'- Embedder: all-MiniLM-L6-v2',
    f'- Retrieval: top-{CFG["k_retrieve"]} by question cosine similarity',
    f'- Gate threshold: {CFG["gate_threshold"]} (admit if `(fg+rg)/2 < {CFG["gate_threshold"]}`)',
    '',
    '## Memory bank sizes',
    '',
    '| Condition | Size | F1 | ΔF1 vs none | 95% CI |',
    '|---|---|---|---|---|',
]
for c in conds:
    r = results[c]
    ci = f'[{r["delta_ci_lo"]:+.3f}, {r["delta_ci_hi"]:+.3f}]' if c != 'none' else '—'
    delta = f'{r["delta_f1"]:+.4f}' if c != 'none' else '—'
    readme_lines.append(f'| {c} | {r["memory_size"]} | {r["f1"]:.4f} | {delta} | {ci} |')

readme_lines += [
    '',
    '## Verdict criteria',
    '',
    '- 🟢 STRONG: ensemble > random > none, paired CI excludes 0',
    '- 🟢 STANDARD: ensemble > none AND ensemble ≥ random',
    '- 🟡 MIXED: ensemble > none but ≈ random',
    '- 🔴 KILL: ensemble ≤ none',
    '',
    '## Files',
    '',
    '- `FINAL_VERDICT.json` — structured verdict + per-condition F1 / EM / CI',
    '- `fig_probe_gated_memory.png` — F1 bars + delta CI',
    '- `train_pool.jsonl` — train trajectories with FG + RG scores',
    '- `test_pool.jsonl` — test generations per (id, condition)',
    '- `memory_banks.json` — bank composition per gate condition',
    '',
    'No LLM judge involved. All scoring is HotpotQA F1 + EM token overlap.',
]
(OUT / 'README.md').write_text('\n'.join(readme_lines))

try:
    api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                      repo_type='dataset', token=HF_TOKEN,
                      commit_message=f'nb47 probe-gated memory: {paper5_status}',
                      allow_patterns=['README.md', 'FINAL_VERDICT.json',
                                      'fig_*.png', 'train_pool.jsonl',
                                      'test_pool.jsonl', 'memory_banks.json'])
    print('✓ pushed')
except Exception as e:
    print(f'HF push failed: {e}')


## Done — interpretation guide

**🟢 STRONG**: Probe gate adds quality on top of memory quantity. Paper-5 gets its strongest claim. Tweet result. Cite ReasoningBank as adjacent work, position probe-gated memory as cost-free upgrade for any memory framework.

**🟢 STANDARD**: Probe-gated memory works, but random memory also works (just less convincingly). Paper-5 claim survives but framed as 'probes are a viable gate, comparable to baseline gates'. ProbePack story holds.

**🟡 MIXED**: Memory helps regardless of gate. Probe-specific value is unclear. Paper-5 dies on this experiment but the broader 'memory frameworks help' finding is preserved. Re-run with stronger probes (DeceptionGuard etc.) when available.

**🔴 KILL**: Memory doesn't help, or probe gate hurts. Paper-5 dies cleanly. Walk back the angle. Honest negative-one-pager possible. ProbePack reverts to single-probe detection product.

Whatever the verdict, push to HF (Phase 8 already does it). Honest negatives strengthen Anthropic-direction more than fake wins.

---

**Free post-hoc supplementary** (if main passes): rebuild memory banks with FG-only / RG-only / max-fusion / bayesian-OR-fusion gates using the same train_pool.jsonl. No new generation needed. Run another retrieval round on test pool to compare.
